In [1]:
import sys
import os
import shutil
import platform
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt;
import pyemu
import flopy
#assert "dependencies" in flopy.__file__
#assert "dependencies" in pyemu.__file__
sys.path.insert(0,"..")
import herebedragons as hbd

In [2]:
org_d = os.path.join('..', '..', 'models', 'monthly_model_files_1lyr_newstress')
tmp_d = os.path.join('freyberg_mf6')
if os.path.exists(tmp_d):
    shutil.rmtree(tmp_d)
shutil.copytree(org_d,tmp_d)
hbd.prep_bins(tmp_d)                                      # get executables                 _____ org_d  ____   tmp_d _____
hbd.prep_deps(tmp_d)                                      # get dependency folders

In [3]:
# os.listdir(tmp_d)                                      `GUI ___PstFrom` 
sim = flopy.mf6.MFSimulation.load(sim_ws=tmp_d)           # simulation          ####   *_ws
gwf = sim.get_model()                                     # flow model
pyemu.os_utils.run("mf6",cwd=tmp_d)                       # run ______ in 'freyberg_mf6' folder
pyemu.os_utils.run(r'mp7 freyberg_mp.mpsim', cwd=tmp_d)   # run modpath7 

loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package ic...
    loading package npf...
    loading package sto...
    loading package oc...
    loading package wel...
    loading package rch...
    loading package ghb...
    loading package sfr...
    loading package obs...
  loading solution package freyberg6...
mf6.exe
mp7.exe freyberg_mp.mpsim


In [4]:
sr = pyemu.helpers.SpatialReference.from_namfile(
        os.path.join(tmp_d, "freyberg6.nam"),                # PstFrom | Flopy modelgrid (alternative) 
        delr=gwf.dis.delr.array, delc=gwf.dis.delc.array)    # sr # Supressed

   could not remove start_datetime


### `PEST Template _________ PstFrom.log `

In [5]:
template_ws = os.path.join("freyberg6_template")           
start_datetime="1-1-2008"
pf = pyemu.utils.PstFrom(original_d=tmp_d,                 # pf = PROJECT
                            new_d=template_ws,             # folder
                            remove_existing=True,           
                            longnames=True,                 
                            spatial_reference=sr,          # spatial reference
                            zero_based=False,              
                            start_datetime=start_datetime, # temporal correlation
                            echo=False)                    

## `Adding Observations OBS`

In [6]:
df = pd.read_csv(os.path.join(template_ws,"heads.csv"),index_col=0)             
hds_df = pf.add_observations("heads.csv",                                
                            insfile="heads.csv.ins",                     # *.csv.INS
                            index_cols="time",                           
                            use_cols=list(df.columns.values),            # index x columns.values
                            prefix="hds")                                # + Weights

df = pd.read_csv(os.path.join(template_ws, "sfr.csv"), index_col=0)      
sfr_df = pf.add_observations("sfr.csv", 
                            insfile="sfr.csv.ins", 
                            index_cols="time", 
                            use_cols=list(df.columns.values), 
                            prefix="sfr")                                

In [17]:
# Before
# os.listdir(template_ws)
# for i in gwf.obs:                                                      # Stored in memory, ... from mf6
#    print(i.output.obs_names

# After
# hds_df.head()
# sfr_df.head()
# [f for f in os.listdir(template_ws) if f.endswith(".ins")]

## `Adding Parameters`

### `Geostatistics`____________________Kh, Kv, Ss, Sy, RCH (xyz,T). GHB, SFR inflow , + WEL pumping uncertain (xyz,T)

In [7]:
v_grid = pyemu.geostats.ExpVario(contribution=1.0,      # sill  
                                    a=1000,             # range 
                                    anisotropy=1.0,    
                                    bearing=0.0         # angle in degrees East of North corresponding to anisotropy ellipse
                                    )
grid_gs = pyemu.geostats.GeoStruct(variograms=v_grid, transform='log')          # _______Grid                   # ax = grid_gs.plot()                                                     
v_pp = pyemu.geostats.ExpVario(contribution=1.0,        
                                    a=15000,           
                                    anisotropy=1.0,    
                                    bearing=0.0         
                                    )
pp_gs = pyemu.geostats.GeoStruct(variograms=v_pp, transform='log')             # _____ Pilot Points             # _ = pp_gs.plot() 
v_time = pyemu.geostats.ExpVario(contribution=1.0,       
                                    a=60, 
                                    anisotropy=1.0, 
                                    bearing=0.0 
                                    )
temporal_gs = pyemu.geostats.GeoStruct(variograms=v_time, transform='none')    # _______Time

### `K __ Option 01 cel by cel __GRID` 

In [21]:
# tag = "npf_k_"
# files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")] # print(files) 

ib = gwf.dis.idomain.array[0]                                       # iDOMAIN ______# plt.imshow(ib)  supressed

In [22]:
f = 'freyberg6.npf_k_layer1.txt'
df_gr = pf.add_parameters(f,                                        # scale _______________________   * CTRL df_gr
                zone_array=ib,                                      
                par_type="grid",                                    
                geostruct=grid_gs,           
                par_name_base=f.split('.')[1].replace("_","")+"gr", # GRID
                pargp=f.split('.')[1].replace("_","")+"gr",         # Group 
                lower_bound=0.2, upper_bound=5.0,                    
                ult_ubound=100, ult_lbound=0.01)                    # Multiplied. _____ Kh cannot exceed these values.multipliers !                    

# df_gr.tail()   # *.tpl  # *.zone 
# [f for f in os.listdir(template_ws) if f.endswith(".tpl")]   

### `K __  Option 02 PILOT POINTS` 

In [23]:
df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    geostruct=pp_gs,
                    par_name_base=f.split('.')[1].replace("_","")+"pp",
                    pargp=f.split('.')[1].replace("_","")+"pp",
                    lower_bound=0.2,upper_bound=5.0,
                    ult_ubound=100, ult_lbound=0.01,
                    pp_space=5)                                      # Unifrom grid ___PILOT POINTS ___ at every 4th row and column

#fig,ax = plt.subplots(1,1,figsize=(2,3))               # PLOT
#ax.set_aspect("equal")
#ax.pcolormesh(sr.xcentergrid, sr.ycentergrid,ib)
#ax.scatter(df_pp.x,df_pp.y)

starting interp point loop for 706 points
starting 0
starting 1
starting 2
starting 3
starting 4
starting 5
starting 6
starting 7
starting 8
starting 9
took 4.122727 seconds


### `K __  Option 03 Cte` 

In [24]:
df_cst = pf.add_parameters(f,                  # Beta _____ Supressed________
                    zone_array=ib,
                    par_type="constant",
                    geostruct=grid_gs,
                    par_name_base=f.split('.')[1].replace("_","")+"cn",
                    pargp=f.split('.')[1].replace("_","")+"cn",
                    lower_bound=0.2,upper_bound=5.0,
                    ult_ubound=100, ult_lbound=0.01)

# [f for f in os.listdir(template_ws) if f.endswith(".tpl")]      
#df = pf.add_observations(f,prefix="hk",zone_array=ib)             
#df  

### `ALl at once` 'sto_ss', 'sto_sy', 'ne_'  

In [25]:
def add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=100, add_coarse=True):   # a function.... for each PAR 
    if isinstance(f,str):
        base = f.split(".")[1].replace("_","")
    else:
        base = f[0].split(".")[1]
        
tag = "sto_ss"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
for f in files[1:]: 
    add_mult_pars(f, lb=0.2, ub=5.0, ulb=1e-7, uub=1e-3)

tag = "sto_sy"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
f = files[0]
add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=0.4)

tag = "ne_"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
for f in files: 
    add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=0.4)                                     #  stored in the inter memory?

# len([f for f in os.listdir(template_ws) if f.endswith(".tpl")])
# [f for f in os.listdir(template_ws) if f.endswith(".tpl")]          

## `Spatial and Temporal Correlation` 

In [ ]:
dts = pd.to_datetime(start_datetime) + pd.to_timedelta(np.cumsum(sim.tdis.perioddata.array["perlen"]),unit='d')
# dts   # the container

In [ ]:
tag = "rch_recharge"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
sp = [int(f.split(".")[1].split('_')[-1]) for f in files]
d = {s:f for s,f in zip(sp,files)}
sp.sort()
files = [d[s] for s in sp]                                                # _____ supressed ______ print(files)
add_mult_pars(files, lb=0.2, ub=5.0, ulb=0, uub=1e-3, add_coarse=False)   # the spatial multiplier parameters; just use the same function
# len([f for f in os.listdir(template_ws) if f.endswith(".tpl")])

In [ ]:
for f in files:                                          # get the stress period number from the file name
    kper = int(f.split('.')[1].split('_')[-1]) - 1       # add the constant parameters (with temporal correlation)
    pf.add_parameters(filenames=f,
                    zone_array=ib,
                    par_type="constant",
                    par_name_base=f.split('.')[1]+"tcn",
                    pargp=f.split('.')[1]+"tcn",
                    lower_bound=0.5, upper_bound=1.5,
                    ult_ubound=1e-3, ult_lbound=0,
                    datetime=dts[kper],                  # this places the parameter value on the "time axis"
                    geostruct=temporal_gs)

## `GHB` Head (+) and Conductance (*)

In [ ]:
tag = "ghb_stress_period_data"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
print(files)

In [ ]:
tag = "ghb_stress_period_data"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
for f in files:
    name = 'ghbcond'                                         
    pf.add_parameters(f,                                      # scale multiplier
                        par_type="grid",                      # GRID
                        geostruct=grid_gs,
                        par_name_base=name+"gr",
                        pargp=name+"gr",
                        index_cols=[0,1,2],                   # column containing lay,row,col
                        use_cols=[4],                         # column containing conductance values
                        lower_bound=0.1,upper_bound=10.0)
    pf.add_parameters(f,
                        par_type="constant",                  # CTE   
                        geostruct=grid_gs,
                        par_name_base=name+"cn",
                        pargp=name+"cn",
                        index_cols=[0,1,2],
                        use_cols=[4],  
                        lower_bound=0.1,upper_bound=10.0,
                        ult_lbound=0.01, ult_ubound=100) #absolute limits
    name = 'ghbhead'   
    pf.add_parameters(f,                                      # Scale additive
                        par_type="grid",                      # GRID
                        geostruct=grid_gs,
                        par_name_base=name+"gr",
                        pargp=name+"gr",
                        index_cols=[0,1,2],
                        use_cols=[3],                         # column containing head values
                        lower_bound=-2.0,upper_bound=2.0,
                        par_style="a",                        # specify additive parameter
                        transform="none",                     
                        ult_lbound=32.5, ult_ubound=42)       # * never lower than the bottom of layer 1
    pf.add_parameters(f,
                        par_type="constant",                  # CTE
                        geostruct=grid_gs,
                        par_name_base=name+"cn",
                        pargp=name+"cn",
                        index_cols=[0,1,2],
                        use_cols=[3],
                        lower_bound=-2.0,upper_bound=2.0, 
                        par_style="a", 
                        transform="none",
                        ult_lbound=32.5, ult_ubound=42) 

## `Pumping rates` (time-invariant) Wel parametrization

 `1º, constant` _____A single multiplier all well pumping rates `2º, grid` multipliers for individual well pumping rates at each stress period

In [ ]:
files = [f for f in os.listdir(template_ws) if "wel_stress_period_data" in f and f.endswith(".txt")]
sp = [int(f.split(".")[1].split('_')[-1]) for f in files]
d = {s:f for s,f in zip(sp,files)}
sp.sort()
files = [d[s] for s in sp]

for f in files:
    kper = int(f.split('.')[1].split('_')[-1]) - 1            # get the stress period number from the file name
    pf.add_parameters(filenames=f,                            # add the constant parameters (with temporal correlation)
                        index_cols=[0,1,2],                   # columns that specify cell location
                        use_cols=[3],                         # columns with parameter values
                        par_type="constant",                  # each well will be adjustable
                        par_name_base="welcst",
                        pargp="welcst", 
                        upper_bound = 4, lower_bound=0.25,
                        datetime=dts[kper],                   # this places the parameter value on the "time axis"
                        geostruct=temporal_gs)
    pf.add_parameters(filenames=f,                            # add the grid parameters; each individual well
                        index_cols=[0,1,2],                   # columns that specify cell location 
                        use_cols=[3],                         # columns with parameter values
                        par_type="grid",                      # each well will be adjustable
                        par_name_base="welgrd",
                        pargp="welgrd", 
                        upper_bound = 4, lower_bound=0.25,
                        datetime=dts[kper])                   # this places the parameter value on the "time axis"                     

## `Conductance` (time-invariant) and `Inflow` (time-variant) SFR parametrization

In [ ]:
tag = "sfr_packagedata"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
assert len(files) == 1                                 # There can be only one! It is tradition. Jokes.
print(files)
f = files[0]                                            # CONDUCTANCE
name = "sfrcond"
pf.add_parameters(f,
                par_type="grid",                        # grid
                geostruct=grid_gs,
                par_name_base=name+"gr",
                pargp=name+"gr",
                index_cols=[0,2,3],
                use_cols=[9],
                lower_bound=0.1,upper_bound=10.0)
pf.add_parameters(f,
                par_type="constant",                    # cte
                geostruct=grid_gs,
                par_name_base=name+"cn",
                pargp=name+"cn",
                index_cols=[0,2,3],
                use_cols=[9],
                lower_bound=0.1,upper_bound=10.0,
                ult_lbound=0.001, ult_ubound=100)      # absolute limits

In [ ]:
files = [f for f in os.listdir(template_ws) if "sfr_perioddata" in f and f.endswith(".txt")]
sp = [int(f.split(".")[1].split('_')[-1]) for f in files]
d = {s:f for s,f in zip(sp,files)}
sp.sort()
files = [d[s] for s in sp]                                     # INFLOW               # print(files) supressed
for f in files:
    kper = int(f.split('.')[1].split('_')[-1]) - 1             # get the stress period number from the file name
    pf.add_parameters(filenames=f,                             # add the parameters   
                        index_cols=[0],                        # reach number
                        use_cols=[2],                          # columns with parameter values
                        par_type="grid",    
                        par_name_base="sfrgr",
                        pargp="sfrgr", 
                        upper_bound = 10, lower_bound=0.1,     # don't need ult_bounds because it is a single multiplier
                        datetime=dts[kper],                    # this places the parameter value on the "time axis"
                        geostruct=temporal_gs)
# [f for f in os.listdir(template_ws) if f.endswith(".tpl")]

### `Initial conditions`

In [ ]:
files = [f for f in os.listdir(template_ws) if "ic_strt" in f and f.endswith(".txt")]
files
for f in files:
    base = f.split(".")[1].replace("_","")
    df = pf.add_parameters(f,par_type="grid",par_style="d",
                      pargp=base,par_name_base=base,upper_bound=50,
                     lower_bound=15,zone_array=ib,transform="none")
    print(df.shape)

### `Forward_run.py`

In [ ]:
pst = pf.build_pst()
# [f for f in os.listdir(template_ws) if f.endswith(".py") or f.endswith(".pst") ]
# _ = [print(line.rstrip()) for line in open(os.path.join(template_ws,"forward_run.py"))]

### `Sys Commands`, __________ command line instruction, stored as a list in `PstFrom.mod_sys_cmds`

In [ ]:
pf.mod_sys_cmds 
pf.mod_sys_cmds.append("mf6")                    # do this only once
pf.mod_sys_cmds
pf.mod_sys_cmds.append("mp7 freyberg_mp.mpsim")  # do this only once
pf.mod_sys_cmds
pst = pf.build_pst()
# _ = [print(line.rstrip()) for line in open(os.path.join(template_ws,"forward_run.py"))]

### `PstFrom.pre_py_cmds` | `PstFrom.post_py_cmds` | `pyemu.os_utils.run()` | `PstFrom.add_py_function()`

In [ ]:
pf.add_py_function("helpers.py","extract_hds_arrays_and_list_dfs()",is_pre_cmd=False)

In [ ]:
import helpers
helpers.test_extract_hds_arrays(template_ws)
# files = [f for f in os.listdir(template_ws) if f.startswith("hdslay")]
# files
for f in files:
    pf.add_observations(f,prefix=f.split(".")[0],obsgp=f.split(".")[0])

In [ ]:
for f in ["inc.csv","cum.csv"]:
    df = pd.read_csv(os.path.join(template_ws,f),index_col=0)
    pf.add_observations(f,index_cols=["totim"],use_cols=list(df.columns.values),
                        prefix=f.split('.')[0],obsgp=f.split(".")[0])

### `Secondary Observations` _____ `helpers.py` custom code for observation processing


In [ ]:
# run the helper function
helpers.process_secondary_obs(ws=template_ws)
# [f for f in os.listdir(template_ws) if f.endswith(".csv")]

In [ ]:
pf.add_py_function("helpers.py",                                               # the function
                    "process_secondary_obs(ws='.')",                           # arguments
                    is_pre_cmd=False)                                          # before | after* the model system command

In [ ]:
df = pd.read_csv(os.path.join(template_ws, "sfr.tdiff.csv"), index_col=0)      # adding OBS to `PstFrom`:
_ = pf.add_observations("sfr.tdiff.csv",                                       # the model output file to read
                            insfile="sfr.tdiff.csv.ins",                       # optional, the instruction file name
                            index_cols="time",                                 # index column header  
                            use_cols=list(df.columns.values),                  # names of columns that include observation values
                            prefix="sfrtd")                                    # prefix to all observation 
                            
df = pd.read_csv(os.path.join(template_ws, "heads.tdiff.csv"), index_col=0)
_ = pf.add_observations("heads.tdiff.csv", 
                            insfile="heads.tdiff.csv.ins", 
                            index_cols="time", 
                            use_cols=list(df.columns.values), 
                            prefix="hdstd") 

### `Re-building the Pst control file`

In [ ]:
pst = pf.build_pst()
# _ = [print(line.rstrip()) for line in open(os.path.join(template_ws,"forward_run.py"))]  # `extract_hds_array_and_list_dfs()`
#obs = pst.observation_data
#obs

### `Observation weights, parameter bounds, transforms, control data, ...`

Call `PstFrom.build_pst()`,     then     `pf.pst.write()`      to OBS custom *.ins files,  W , parameter group INCTYP's and forecasts.

#### Add Observations from INS files

Recall that we wish to include observations of particle end time and status. As mentioned earlier, MP7 output files are not ina nicely organized tabular format - so we need to construct a custom instruction file. We will do this now:

In [ ]:
# write a really simple instruction file to read the MODPATH end point file
out_file = "freyberg_mp.mpend"
ins_file = out_file + ".ins"
with open(os.path.join(template_ws, ins_file),'w') as f:
    f.write("pif ~\n")
    f.write("l7 w w w w !part_status! w w !part_time!\n")

Now add these observations to the `Pst`:

In [ ]:
pst.add_observations(ins_file=os.path.join(template_ws, ins_file),
                    out_file=os.path.join(template_ws, out_file),
                            pst_path='.')

# and then check what changed                            
obs = pst.observation_data
obs.loc[obs.obsnme=='part_status', 'obgnme'] = 'part'
obs.loc[obs.obsnme=='part_time', 'obgnme'] = 'part'

obs.iloc[-2:]

#### Parameters with Zero as Intial Value

Recall that we assigned additive parameters to the GHB heads. Our initial parameter values for these parameter types were set as 0 (zero). This creates a wee bit of trouble when calculating derivatives - this is only an issue for algorithms that use finite-difference derivatives. There are a couple of ways we could get around it. One way is to add an "offset" to the parameter initial values and to the parameter bounds. Another is to use "absolute" increment types (INCTYP). See the PEST manual or PEST++ user guides for descriptions of increment types. 

We will apply both here. 

We will assign INCTYP as 'absolute'. We will leave DERINC as 0.01 (the default). It is a reasonable value in this case.

In [ ]:
head_pargps = [i for i in pst.adj_par_groups if 'head' in i]
head_pargps

In [ ]:
pst.parameter_groups.loc[head_pargps, 'inctyp'] = 'absolute'

Now add the "offset" to parameter data entries:

In [ ]:
par = pst.parameter_data
par_names = par.loc[par.parval1==0].parnme

par.loc[par_names].head()

In [ ]:
offset = -10
par.loc[par_names, 'offset'] = offset
par.loc[par_names, ['parval1', 'parlbnd', 'parubnd']] -= offset

par.loc[par_names].head()

#### Forecasts

For most models there is a forecast/prediction that someone needs. Rather than waiting until the end of the project, the forecast should be entered into your thinking and workflow __right at the beginning__.  Here we do this explicitly by monitoring the forecasts as "observations" in the control file.  This way, for every PEST(++) analysis we do, we can watch what is happening to the forecasts - #winning

The optional PEST++ `++forecasts` control variable allows us to provide the names of one or more observations featured in the "observation data" section of the PEST control file; these are treated as predictions in FOSM predictive uncertainty analysis by PESTPP-GLM. It is also a convenient way to keep track of "forecast" observations (makes post-processing a wee bit easier later on).

Recall that, for our synthetic case we are interested in forecasting:

 - groundwater level in the upper layer at row 9 and column 1 (site named "trgw-0-9-1") in stress period 22 (time=640);
 - the "tailwater" surface-water/groundwater exchange during stress period 13 (time=367); and
 - the "headwater" surface-water/groundwater exchange at stress period 22 (time=640).
 - the particle travel time.


In [ ]:
forecasts =[
            'oname:sfr_otype:lst_usecol:tailwater_time:4383.5',
            'oname:sfr_otype:lst_usecol:headwater_time:4383.5',
            'oname:hds_otype:lst_usecol:trgw-0-9-1_time:4383.5',
            'part_time'
            ]

forecasts

In [ ]:
fobs = obs.loc[forecasts,:]
fobs

We will just set this optional pest++ argument because it will trigger certain automatic behavior later in PESTPP-GLM

In [ ]:
pst.pestpp_options['forecasts'] = forecasts

### Re-write the Control File!

Make sure to re-**write** the PEST control file. But beware, if you re-**build** the `Pst`, all these changes will be lost.

In [ ]:
pst.write(os.path.join(template_ws, 'freyberg_mf6.pst'),version=2)

So that was pretty epic. We now have a (very) high-dimensional PEST interface that includes secondary observations, as well as forecasts, ready to roll. 

If you inspect the folder, you will see PEST control file and all the necessary instruction and template files. Because we have >10k parameters, version 2 of the PEST control file was written by default. 

Shall we check that it works? Let's run PEST once (i.e. with NOPTMAX=0). Now, by default, noptmax is set to zero. But just to check:

In [ ]:
pst.control_data.noptmax

OK, so when we run PEST it will call the model once and then stop. If the next cell is successful, then everything is working. Check the folder, you should see PEST output files. (We will go into these and how to process PEST outcomes in subsequent tutorials).

In [ ]:
pyemu.os_utils.run('pestpp-glm freyberg_mf6.pst', cwd=template_ws)

Recall that we assigned observation values generated from the "base model run"? If we setup everything correctly, this means that PEST should have obtained residuals very close to zero. As mentioned, this is a good way to check for problems early on.

Let's check the Phi recorded in the *.iobj file (could also check the *.rec or *.rei files).

In [ ]:
# read the file
iobj = pd.read_csv(os.path.join(template_ws, 'freyberg_mf6.iobj'))

# check value in phi column
iobj.total_phi

 Sweet! Zero. All is well.

### Prior Parameter Covariance Matrix

One the major reasons `PstFrom` was built is to help with building the Prior - both covariance matrix and ensemble - with geostatistical correlation.  Remember all that business above related to geostatistical structures and correlations?  This is where it pays off.

Let's see how this works.  For cases with less than about 30,000 parameters, we can actually generate and visualize the prior parameter covariance matrix.  If you have more parameters, this matrix may not fit in memory.  But, not to worry, `PstFrom` has some trickery to help generate the geostatistical prior ensemble even in cases where the number of parameters is greater than 30,000. 


In [ ]:
# build the prior covariance matrix and store it as a compresed bianry file (otherwise it can get huge!)
# depending on your machine, this may take a while...
if pf.pst.npar < 35000:  #if you have more than about 35K pars, the cov matrix becomes hard to handle
    cov = pf.build_prior(fmt='coo', filename=os.path.join(template_ws,"prior_cov.jcb"))
    # and take a peek at a slice of the matrix
    try: 
        x = cov.x.copy()
        x[x==0] = np.NaN
        plt.imshow(x[:1000,:1000])
    except:
        pass

snap!  That big block must be a grid-scale parameter group...

In [ ]:
cov.row_names[:10]

And now generate a prior parameter ensemble. This step is relevant for using pestpp-ies in subsequent tutorials. Note: you do not have to call `build_prior()` before calling `draw()`!

In [ ]:
pe = pf.draw(num_reals=1000, use_specsim=True) # draw parameters from the prior distribution
pe.enforce() # enforces parameter bounds
pe.to_binary(os.path.join(template_ws,"prior_pe.jcb")) #writes the paramter ensemble to binary file
assert pe.shape[1] == pst.npar

Let's now test-run one of these geostatistical realizations (always a good idea!).  We do this by replacing the `parval1` values in the control with a row from `pe`:

In [ ]:
pst.parameter_data.loc[:,"parval1"] = pe.loc[pe.index[0],pst.par_names].values
pst.parameter_data.parval1.values

In [ ]:
pst.control_data.noptmax = 0
pst.write(os.path.join(template_ws,"test.pst"),version=2)
pyemu.os_utils.run("pestpp-glm test.pst",cwd=template_ws)

If all went well, that's it! The PEST-interface is setup, tested and we have our prior prepared. We should be good to go!

### Bonus: Understanding Multiplier-Parameters

Now the multiplier files in the `template_ws/mult` folder and the MF6 input files in the `template_ws` folder contain the values corresponding to this realization, so we can visualize the multiplier parameter process:

In [ ]:
df = pd.read_csv(os.path.join(template_ws,"mult2model_info.csv"))
kh1_df = df.loc[df.model_file.str.contains("npf_k_layer1"),:]
kh1_df

In [ ]:
org_arr = np.loadtxt(os.path.join(template_ws,kh1_df.org_file.iloc[0]))
inp_arr = np.loadtxt(os.path.join(template_ws,kh1_df.model_file.iloc[0]))
mlt_arrs = [np.loadtxt(os.path.join(template_ws,afile)) for afile in kh1_df.mlt_file]
arrs = [org_arr]
arrs.extend(mlt_arrs)
arrs.append(inp_arr)
names = ["org"]
names.extend([mf.split('.')[0].split('_')[-1] for mf in kh1_df.mlt_file])
names.append("MF6 input")
fig,axes = plt.subplots(1,kh1_df.shape[0]+2,figsize=(5*kh1_df.shape[0]+2,5))
for i,ax in enumerate(axes.flatten()):
    arr = np.log10(arrs[i])
    arr[ib==0] = np.NaN
    cb = ax.imshow(arr)
    plt.colorbar(cb,ax=ax)
    ax.set_title(names[i],loc="left")
plt.tight_layout()    
    

So now we can see the difference the pilot point geostructure makes compared to the grid-scale geostructure.  The pilot point array has a much stronger spatial correlation (over a larger distance) than the grid-scale...

# Finally, what's up with all these new files and folders? We can trace through them to see how multipliers are applied

In [ ]:
from IPython.display import Image, display
display(Image(url='./pest_flow2.gif'))

### BETA EXTRA `ALl at once`  'npf_k_ above',  + + +   

def add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=100, add_coarse=True):   # a function.... for each PAR 
    if isinstance(f,str):
        base = f.split(".")[1].replace("_","")
    else:
        base = f[0].split(".")[1]
    pf.add_parameters(f,                                        
                    zone_array=ib,
                    par_type="grid", #_________________________ # grid (fine) scale parameters
                    geostruct=grid_gs,  
                    par_name_base=base+"gr", 
                    pargp=base+"gr", 
                    lower_bound=lb, upper_bound=ub, 
                    ult_ubound=uub, ult_lbound=ulb)
    pf.add_parameters(f,                                        
                        zone_array=ib,
                        par_type="pilotpoints", #______________ # pilot point (medium) scale parameters
                        geostruct=pp_gs,
                        par_name_base=base+"pp",
                        pargp=base+"pp",
                        lower_bound=lb, upper_bound=ub,
                        ult_ubound=uub, ult_lbound=ulb,
                        pp_space=5) 
    if add_coarse==True:
        pf.add_parameters(f,                                    
                            zone_array=ib,
                            par_type="constant", #______________ # constant (coarse) scale parameters
                            geostruct=grid_gs,
                            par_name_base=base+"cn",
                            pargp=base+"cn",
                            lower_bound=lb, upper_bound=ub,
                            ult_ubound=uub, ult_lbound=ulb)
    return